In [ ]:
import os
import torch
import argparse
import sys
from pathlib import Path
from PIL import Image
from torchvision import transforms
from typing import List, Dict
from datetime import datetime

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(project_root)

from models import get_model


In [ ]:
# Preprocessing shared by all models
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

label_names = ["Real", "Fake"]

In [ ]:
def load_images(path):
    """Load one image OR all images in a folder."""
    p = Path(path)
    
    if p.is_file():
        return [p]
    elif p.is_dir():
        images = list(p.glob("*.jpg")) + list(p.glob("*.png"))
        if len(images) == 0:
            raise ValueError("No .jpg or .png images found in folder.")
        return images
    else:
        raise ValueError(f"Invalid path: {path}")


def load_checkpoint(model, ckpt_path, device):
    """Load weights into the model."""
    state_dict = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(state_dict)
    model.eval()
    return model


def infer_one(model, img_tensor, device):
    """Run inference on a single tensor."""
    with torch.no_grad():
        out = model(img_tensor.unsqueeze(0).to(device))
        probs = torch.softmax(out, dim=1)[0]
        pred_idx = probs.argmax().item()
    return pred_idx, probs.cpu().numpy()


def run_inference(input_path, model_ckpts, device="cuda"):
    """
    Notebook-friendly inference controller.

    Parameters:
        input_path: str — path to image or folder
        model_ckpts: dict — { model_name: [list of ckpt paths] }
        device: "cuda" or "cpu"

    Returns:
        results (list of dicts)
        results_file (txt file name)
    """

    device = torch.device(device if torch.cuda.is_available() else "cpu")

    # Load images
    image_paths = load_images(input_path)

    # Prepare output file
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    results_file = f"results_{timestamp}.txt"
    open(results_file, "w").close()

    def log(text):
        print(text)
        with open(results_file, "a") as f:
            f.write(text + "\n")

    log(f"Device: {device}")
    log(f"Loaded {len(image_paths)} images.")
    log("Starting inference...\n")

    all_results = []

    #   MAIN LOOP OVER MODELS AND THEIR CHECKPOINTS
    for model_name, ckpt_list in model_ckpts.items():

        if not ckpt_list:
            continue  # skip empty lists

        log(f"\n=== MODEL: {model_name} ===")

        # Build base architecture ONCE
        base_model = get_model(model_name, zero_init=False).to(device)

        for ckpt_path in ckpt_list:
            log(f"\n  Using checkpoint: {ckpt_path}")
            model = load_checkpoint(base_model, ckpt_path, device)

            # -------------------------------
            # RUN MODEL ON EACH IMAGE
            # -------------------------------
            for img_path in image_paths:
                img = Image.open(img_path).convert("RGB")
                img_tensor = preprocess(img)

                pred_idx, probs = infer_one(model, img_tensor, device)
                pred_label = label_names[pred_idx]

                line = (f"    {img_path.name}: {pred_label} "
                        f"(Real={probs[0]:.4f}, Fake={probs[1]:.4f})")
                log(line)

                all_results.append({
                    "model": model_name,
                    "checkpoint": ckpt_path,
                    "image": img_path.name,
                    "pred_idx": pred_idx,
                    "pred_label": pred_label,
                    "prob_real": float(probs[0]),
                    "prob_fake": float(probs[1]),
                })

    log("\nInference complete.")
    return all_results, results_file


In [ ]:
model_ckpts = {
    "resnet50": ["training/NEW_Temporal_ResNet50_run_20251201_173932.pth"],
    "vit": ["training/NEW_Temporal_ViT_run_20251201_161225.pth"],
    "resnet50_fft": ["training/lNEW_nonzero_Temporal_ResNet50+FFT_run_20251202_020614.pth"],
    "vit_fft": ["training/NEW_Temporal_ViT_FFT_run_20251201_161700.pth"]
}

results, results_file = run_inference(
    input_path="test_images/",
    model_ckpts=model_ckpts,
    device="cuda"
)
